# Visium HD Spatial Transcriptomics Quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/stevenpastor/spatial_transcriptomics_resources/blob/main/notebook/visium_hd_quickstart.ipynb)

Condensed walkthrough of Visium HD analysis on the **10x Genomics Human CRC Patient 1** dataset ([Nature Genetics 2025](https://www.nature.com/articles/s41588-025-02193-3)). For detailed explanations, see the [comprehensive tutorial](https://colab.research.google.com/github/stevenpastor/spatial_transcriptomics_resources/blob/main/notebook/visium_hd_crc_p1_tutorial.ipynb).

---

## Setup

In [ ]:
# Package installation + data download
import os, sys, subprocess
from pathlib import Path

# Install packages
print("Installing packages...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "scanpy>=1.10", "squidpy>=1.6", "anndata>=0.10",
    "numpy>=1.24", "pandas>=2.0", "scipy>=1.11",
    "matplotlib>=3.7", "seaborn>=0.13",
    "scikit-image>=0.21", "Pillow>=10",
    "igraph", "leidenalg",
    "tqdm>=4.65", "requests>=2.31", "h5py>=3.9", "pyarrow>=13",
])
print("Done.")

# Download precomputed data from Figshare (per-file via API, no zip)
FIGSHARE_ARTICLE_ID = "31937262"
FIGSHARE_API = f"https://api.figshare.com/v2/articles/{FIGSHARE_ARTICLE_ID}/files"

DATA_DIR = Path("/content/data")
PRECOMPUTED_DIR = DATA_DIR / "precomputed"
SPATIAL_DIR = PRECOMPUTED_DIR / "spatial"
PRECOMPUTED_DIR.mkdir(parents=True, exist_ok=True)
SPATIAL_DIR.mkdir(parents=True, exist_ok=True)

RAW_H5AD = PRECOMPUTED_DIR / "crc_p1_raw_50k.h5ad"
ANNOT_H5AD = PRECOMPUTED_DIR / "crc_p1_annotated_50k.h5ad"

SPATIAL_FILENAMES = {
    "tissue_hires_image.png",
    "tissue_lowres_image.png",
    "scalefactors_json.json",
}

def _target_path(name):
    return SPATIAL_DIR / name if name in SPATIAL_FILENAMES else PRECOMPUTED_DIR / name

def download_file(url, dest, desc="Downloading"):
    import requests
    from tqdm.auto import tqdm
    tmp = dest.with_suffix(dest.suffix + ".part")
    with requests.get(url, stream=True, timeout=(60, None)) as resp:
        resp.raise_for_status()
        total = int(resp.headers.get("content-length", 0))
        with open(tmp, "wb") as f, tqdm(total=total, unit="B", unit_scale=True, desc=desc) as pbar:
            for chunk in resp.iter_content(chunk_size=8 * 1024 * 1024):
                if chunk:
                    f.write(chunk)
                    pbar.update(len(chunk))
    tmp.replace(dest)
    print(f"Saved: {dest.relative_to(DATA_DIR)} ({dest.stat().st_size / 1e6:.2f} MB)")

def fetch_figshare_files():
    import requests
    resp = requests.get(FIGSHARE_API, timeout=60)
    resp.raise_for_status()
    return resp.json()

needed = [
    RAW_H5AD,
    ANNOT_H5AD,
    SPATIAL_DIR / "tissue_hires_image.png",
    SPATIAL_DIR / "tissue_lowres_image.png",
    SPATIAL_DIR / "scalefactors_json.json",
]

if not all(p.exists() for p in needed):
    print("Downloading precomputed data from Figshare (per file)...")
    for entry in fetch_figshare_files():
        name = entry["name"]
        dest = _target_path(name)
        expected_size = entry.get("size")
        if dest.exists() and expected_size is not None and dest.stat().st_size == expected_size:
            print(f"Skipping (already present): {dest.relative_to(DATA_DIR)}")
            continue
        download_file(entry["download_url"], dest, desc=name)
    print("Download complete.")
else:
    print("Precomputed data already downloaded.")

# Clone repo for utils.py
REPO_URL = "https://github.com/stevenpastor/spatial_transcriptomics_resources.git"
REPO_DIR = Path("/content/spatial_tutorial")
if not REPO_DIR.exists():
    print("Cloning tutorial repo (for utility functions)...")
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)])
SCRIPTS_DIR = REPO_DIR / "scripts"
print("Ready.")

In [ ]:
import sys, os, time, warnings, gc
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import squidpy as sq
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import sparse

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=100, facecolor="white", frameon=False, fontsize=12)

# Paths
DATA_DIR = Path("/content/data")
PRECOMPUTED_DIR = DATA_DIR / "precomputed"
FIG_DIR = Path("/content/figures")
FIG_DIR.mkdir(exist_ok=True)

sys.path.insert(0, str(SCRIPTS_DIR))
from utils import (
    load_visium_hd_binned,
    compute_qc_metrics,
    spatial_qc_plots,
    spatial_outlier_detection,
)

print(f"scanpy {sc.__version__}, squidpy {sq.__version__}")

## Data Loading

We load a random subsample of ~50K bins from the full ~500K+ 8 um binned CRC Patient 1 dataset. The subsample was created to keep file sizes under GitHub's 100 MB limit (see `scripts/generate_precomputed.py`). Ground-truth annotations from the original study group are embedded in the file.

In [ ]:
%%time
# List precomputed files
for f in sorted(PRECOMPUTED_DIR.iterdir()):
    size_mb = f.stat().st_size / 1e6 if f.is_file() else 0
    print(f"  {f.name:40s}  {size_mb:6.1f} MB" if f.is_file() else f"  {f.name}/")

# Load 8 um binned data (random subsample of ~50K bins from the full dataset)
adata_8um = sc.read_h5ad(PRECOMPUTED_DIR / "crc_p1_raw_50k.h5ad")
print(f"\nLoaded: {adata_8um.shape[0]:,} bins x {adata_8um.shape[1]:,} genes")
print(f"Spatial range X: [{adata_8um.obsm['spatial'][:,0].min():.0f}, {adata_8um.obsm['spatial'][:,0].max():.0f}]")
print(f"Spatial range Y: [{adata_8um.obsm['spatial'][:,1].min():.0f}, {adata_8um.obsm['spatial'][:,1].max():.0f}]")
if sparse.issparse(adata_8um.X):
    print(f"Median UMIs/bin: {np.median(adata_8um.X.sum(axis=1).A1):.0f}")
    print(f"Median genes/bin: {np.median((adata_8um.X > 0).sum(axis=1).A1):.0f}")

# Ground-truth annotations (from the original study group, not us)
if "UnsupervisedL1" in adata_8um.obs.columns:
    print(f"\nGround-truth annotations present: {adata_8um.obs['UnsupervisedL1'].notna().sum():,} / {adata_8um.shape[0]:,} bins")
    print(adata_8um.obs["UnsupervisedL1"].value_counts().to_string())

## Bins Outside Tissue

The Visium HD capture area is a 6.5 x 6.5 mm square, but the tissue section rarely covers the entire slide. Space Ranger flags each bin as `in_tissue` (1) or off-tissue (0) based on whether it falls under the tissue section. Off-tissue bins capture only ambient RNA and background noise, not real signal, so they must be removed before analysis.

Our loading function (`load_visium_hd_binned` in `utils.py`) automatically filters to `in_tissue == 1`. The full dataset has ~500K+ bins on the entire capture area, but only ~260K fall under tissue. Our precomputed file is a random subsample of those in-tissue bins.

Below we verify that all bins in our data are in-tissue and visualize the tissue boundary by plotting bin positions against the full capture area extent.

In [ ]:
# Verify in_tissue filtering and show tissue boundary
coords = adata_8um.obsm["spatial"]

# Check if in_tissue column exists (it was used for filtering, may or may not be retained)
if "in_tissue" in adata_8um.obs.columns:
    print(f"in_tissue values: {adata_8um.obs['in_tissue'].value_counts().to_dict()}")
    print("All bins are in_tissue == 1 (off-tissue bins were removed during loading)")
else:
    print("in_tissue column was dropped after filtering (all remaining bins are in-tissue)")

print(f"\nBins in dataset: {adata_8um.shape[0]:,}")
print(f"Coordinate range: X=[{coords[:,0].min():.0f}, {coords[:,0].max():.0f}], "
      f"Y=[{coords[:,1].min():.0f}, {coords[:,1].max():.0f}]")

# Visualize the tissue footprint against the full capture area
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: all bin positions colored by total counts
total_cts = np.asarray(adata_8um.X.sum(axis=1)).flatten() if sparse.issparse(adata_8um.X) else adata_8um.X.sum(axis=1)
sc_plot = axes[0].scatter(
    coords[:, 0], coords[:, 1], s=0.2, c=total_cts,
    cmap="viridis", edgecolors="none", rasterized=True,
)
axes[0].set_aspect("equal")
axes[0].invert_yaxis()
axes[0].set_title("In-tissue bins (colored by UMI count)")
axes[0].axis("off")
plt.colorbar(sc_plot, ax=axes[0], shrink=0.6, label="Total UMIs")

# Right: highlight bins that would be at risk of removal by QC
# Bins with very low counts are near the tissue edge or in sparse regions
low_count_thresh = np.percentile(total_cts, 5)
is_edge = total_cts < low_count_thresh

axes[1].scatter(
    coords[~is_edge, 0], coords[~is_edge, 1], s=0.2,
    c="steelblue", alpha=0.3, edgecolors="none", rasterized=True, label="Interior bins",
)
axes[1].scatter(
    coords[is_edge, 0], coords[is_edge, 1], s=0.5,
    c="red", alpha=0.6, edgecolors="none", rasterized=True,
    label=f"Low-count bins (<{low_count_thresh:.0f} UMIs, bottom 5%)",
)
axes[1].set_aspect("equal")
axes[1].invert_yaxis()
axes[1].set_title("Edge and low-count bins")
axes[1].axis("off")
axes[1].legend(markerscale=10, fontsize=10, loc="lower right")

plt.tight_layout()
plt.savefig(FIG_DIR / "crc_tissue_boundary.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nLow-count edge bins (bottom 5%): {is_edge.sum():,} / {len(is_edge):,}")
print("These are often at the tissue boundary where bins partially overlap the edge.")
print("They get removed downstream by the per-bin QC filters (min total_counts threshold).")

## H&E Image Verification

Overlay bin positions on the H&E to check that spatial coordinates are correctly registered to the histology.

In [ ]:
%%time
# Load H&E and overlay bin positions
import json as _json

if "spatial_image" in adata_8um.uns:
    he_image = adata_8um.uns["spatial_image"]
    scalefactors = adata_8um.uns.get("scalefactors", {})
    # Embedded image is lowres, so use the lowres scale factor
    sf = scalefactors.get("tissue_lowres_scalef", scalefactors.get("tissue_hires_scalef", 1.0))
    print(f"Image from adata.uns, shape={he_image.shape}, sf={sf}")
else:
    from skimage import io as skio
    spatial_dir = PRECOMPUTED_DIR / "spatial"
    he_image = None
    for name in ["tissue_hires_image.png", "tissue_lowres_image.png"]:
        candidate = spatial_dir / name
        if candidate.exists():
            he_image = skio.imread(str(candidate))
            with open(spatial_dir / "scalefactors_json.json") as f:
                scalefactors = _json.load(f)
            # Match scale factor to whichever image was loaded
            key = "tissue_hires_scalef" if "hires" in name else "tissue_lowres_scalef"
            sf = scalefactors.get(key, 1.0)
            break

if he_image is not None:
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    # Left: H&E only
    axes[0].imshow(he_image)
    axes[0].set_title("H&E Image")
    axes[0].axis("off")

    # Right: bins overlaid on semi-transparent H&E
    coords = adata_8um.obsm["spatial"]
    axes[1].imshow(he_image, alpha=0.3)
    axes[1].scatter(
        coords[:, 0] * sf, coords[:, 1] * sf,
        s=0.3, c="red", alpha=0.5, rasterized=True,
    )
    # Lock axes to the image extent so the H&E isn't shrunk to an inset
    axes[1].set_xlim(0, he_image.shape[1])
    axes[1].set_ylim(he_image.shape[0], 0)
    axes[1].set_title("Bins overlaid on H&E (semi-transparent)")
    axes[1].axis("off")

    plt.tight_layout()
    plt.savefig(FIG_DIR / "crc_image_verification.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Bins plotted: {coords.shape[0]:,}")
else:
    print("H&E image not found. Plotting spatial coordinates only.")
    fig, ax = plt.subplots(figsize=(8, 7))
    coords = adata_8um.obsm["spatial"]
    total_counts = np.asarray(adata_8um.X.sum(axis=1)).flatten() if sparse.issparse(adata_8um.X) else adata_8um.X.sum(axis=1)
    sc_plot = ax.scatter(coords[:, 0], coords[:, 1], s=0.1,
                         c=total_counts, cmap="viridis", edgecolors="none", rasterized=True)
    ax.set_aspect("equal"); ax.invert_yaxis(); ax.axis("off")
    ax.set_title("Bin positions (colored by UMI count)")
    plt.colorbar(sc_plot, ax=ax, shrink=0.6)
    plt.tight_layout()
    plt.show()

### What this means

* The left panel shows the H&E histology image. The right panel overlays all bin positions (red dots) on a semi-transparent version of the same image. The dots should trace the tissue boundary with no offset, rotation, or mirroring.

* The dots look sparse because this is a ~50K random subsample of ~500K+ total bins. Gaps in coverage are from subsampling, not data loss.

## Per-Bin Quality Metrics

First, we compute standard QC metrics for each bin and check their distributions. This is the same kind of QC you would do for scRNA-seq, applied to spatial bins.

| Metric | What it measures | High values suggest |
|--------|-----------------|-------------------|
| `total_counts` | Total UMIs per bin | Dense tissue / doublets |
| `n_genes_by_counts` | Genes detected per bin | Transcriptomic complexity |
| `pct_counts_mt` | % mitochondrial UMIs | Dying/stressed cells |
| `complexity` | log(genes)/log(counts) | Transcriptomic diversity |

In [ ]:
%%time
# Compute QC metrics
adata_8um = compute_qc_metrics(adata_8um)

print("QC metric summary:")
for col in ["total_counts", "n_genes_by_counts", "pct_counts_mt", "complexity"]:
    vals = adata_8um.obs[col]
    print(f"  {col:25s}  median={vals.median():8.1f}  mean={vals.mean():8.1f}  "
          f"[{vals.min():.1f}, {vals.max():.1f}]")

# Violin plots
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, metric in zip(axes, ["total_counts", "n_genes_by_counts",
                              "pct_counts_mt", "complexity"]):
    parts = ax.violinplot(adata_8um.obs[metric].dropna().values, showmedians=True, showextrema=False)
    for pc in parts["bodies"]:
        pc.set_facecolor("steelblue"); pc.set_alpha(0.7)
    ax.set_title(metric.replace("_", "\n"), fontsize=10)
    ax.set_xticks([])
plt.suptitle("QC Distributions (8 um bins)", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / "crc_qc_violins.png", dpi=150, bbox_inches="tight")
plt.show()

# Scatter: genes vs UMIs colored by mt%
fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(
    adata_8um.obs["total_counts"], adata_8um.obs["n_genes_by_counts"],
    c=adata_8um.obs["pct_counts_mt"], s=0.5, cmap="RdYlBu_r",
    edgecolors="none", rasterized=True,
    vmin=0, vmax=adata_8um.obs["pct_counts_mt"].quantile(0.95),
)
ax.set_xlabel("Total UMI counts"); ax.set_ylabel("Genes detected")
ax.set_title("Gene complexity colored by mitochondrial %")
plt.colorbar(scatter, ax=ax, label="pct_counts_mt")
plt.tight_layout()
plt.savefig(FIG_DIR / "crc_qc_scatter.png", dpi=150, bbox_inches="tight")
plt.show()

# Histograms with data-driven thresholds
min_counts = max(adata_8um.obs["total_counts"].quantile(0.01), 10)
max_counts = adata_8um.obs["total_counts"].quantile(0.995)
min_genes = max(adata_8um.obs["n_genes_by_counts"].quantile(0.01), 5)
max_mt = min(adata_8um.obs["pct_counts_mt"].quantile(0.95), 25)

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
thresholds = [
    ("total_counts", min_counts, max_counts, "UMI Counts"),
    ("n_genes_by_counts", min_genes, None, "Genes Detected"),
    ("pct_counts_mt", None, max_mt, "Mitochondrial %"),
    ("complexity", None, None, "Complexity"),
]
for ax, (metric, lo, hi, label) in zip(axes, thresholds):
    ax.hist(adata_8um.obs[metric].dropna(), bins=100, color="steelblue", alpha=0.7, edgecolor="none")
    if lo is not None:
        ax.axvline(lo, color="red", ls="--", lw=1.5, label=f"min={lo:.1f}")
    if hi is not None:
        ax.axvline(hi, color="red", ls="--", lw=1.5, label=f"max={hi:.1f}")
    ax.set_title(label)
    if lo is not None or hi is not None:
        ax.legend(fontsize=8)
plt.suptitle("QC Histograms with Suggested Thresholds", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / "crc_qc_histograms.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Suggested thresholds: min_counts={min_counts:.1f}, max_counts={max_counts:.1f}, "
      f"min_genes={min_genes:.1f}, max_mt={max_mt:.1f}%")

### What this means

* **UMIs/bin**: Median ~147. Expected for 8 um resolution where each bin covers a tiny area. The right-skewed tail reflects bins over dense or transcriptionally active regions.

* **Mitochondrial %**: Median 6.7%, but a long tail to 100%. Bins at 100% mt are dead/damaged cells or ambient mitochondrial RNA. Why do tumor cells often have high mt%? Tumor cells are metabolically hyperactive and often grow in hypoxic conditions, which upregulates mitochondrial gene expression. So some elevated mt% in tumor regions is biological, not damage. The key is to check spatially: if high-mt bins cluster at tissue edges and folds, that is damage and should be filtered. If they are dispersed through the tumor core, that is likely real biology.

* **Scatter plot**: High-mt bins (red/yellow) fall below the diagonal, meaning high counts but few unique genes. This is the classic damage signature. Supports using both mt% and gene count filters together.

* These global metrics flag problematic bins, but they treat every bin the same regardless of location. Next we add spatial QC to catch location-dependent artifacts.

## Spatial QC

Global thresholds treat every bin identically. A bin with 50 UMIs in dense tumor is suspicious, but the same bin in sparse stroma is normal. Spatial QC compares each bin to its neighbors and flags bins that deviate from their local context. This catches tissue folds, air bubbles, and edge artifacts that global metrics miss.

In [ ]:
%%time
# Spatial QC heatmaps
spatial_qc_plots(
    adata_8um,
    metrics=["total_counts", "n_genes_by_counts", "pct_counts_mt", "complexity"],
    figsize=(22, 4), spot_size=0.3,
    save=str(FIG_DIR / "crc_spatial_qc_heatmaps.png"),
)

# Spatial outlier detection: compare each bin to its 20 nearest neighbors
print("Computing spatial neighbors (k=20)...")
sq.gr.spatial_neighbors(adata_8um, n_neighs=20, coord_type="generic")

outlier_metrics = ["total_counts", "pct_counts_mt", "n_genes_by_counts"]
total_outliers = np.zeros(adata_8um.shape[0], dtype=bool)
for metric in outlier_metrics:
    outliers = spatial_outlier_detection(adata_8um, metric=metric, z_threshold=3.0)
    n_out = outliers.sum()
    print(f"  {metric:25s}: {n_out:,} outliers ({100*n_out/len(outliers):.2f}%)")
    total_outliers = total_outliers | outliers.values

adata_8um.obs["spatial_outlier"] = total_outliers
print(f"\nTotal spatial outliers: {total_outliers.sum():,} ({100*total_outliers.mean():.2f}%)")

fig, ax = plt.subplots(figsize=(8, 7))
coords = adata_8um.obsm["spatial"]
ax.scatter(coords[~total_outliers, 0], coords[~total_outliers, 1],
           s=0.1, c="lightgray", alpha=0.3, rasterized=True, label="Normal")
ax.scatter(coords[total_outliers, 0], coords[total_outliers, 1],
           s=1, c="red", alpha=0.6, rasterized=True, label="Outlier")
ax.set_aspect("equal"); ax.invert_yaxis(); ax.axis("off")
ax.legend(markerscale=10, fontsize=10)
ax.set_title(f"Spatial outliers (n={total_outliers.sum():,})")
plt.tight_layout()
plt.savefig(FIG_DIR / "crc_spatial_outliers.png", dpi=150, bbox_inches="tight")
plt.show()

### What this means

* **Spatial heatmaps**: Total counts and genes show a gradient with higher signal in the dense tumor region and lower signal in the stroma. Mitochondrial % is diffuse rather than concentrated at edges, suggesting a mix of biology and tissue quality.

* **Spatial outlier detection**: For each bin, we compare its QC metrics to its 20 nearest spatial neighbors. Bins deviating >3 standard deviations are flagged. About 4% of bins are outliers, scattered across the tissue rather than clustered in one spot.

* **Why do both?** Global QC removes bins that are bad everywhere (near-zero UMIs, 100% mt). Spatial QC removes bins that look fine globally but are anomalous relative to their surroundings (e.g., a low-count bin surrounded by high-count neighbors, likely a capture artifact). Together they catch more problems than either alone.

## Filtering and Normalization

In [ ]:
%%time
# Apply global + spatial QC filters
n_before = adata_8um.shape[0]

min_counts = max(adata_8um.obs["total_counts"].quantile(0.01), 10)
max_counts = adata_8um.obs["total_counts"].quantile(0.995)
min_genes = max(adata_8um.obs["n_genes_by_counts"].quantile(0.01), 5)
max_mt = min(adata_8um.obs["pct_counts_mt"].quantile(0.95), 25)

keep = (
    (adata_8um.obs["total_counts"] >= min_counts) &
    (adata_8um.obs["total_counts"] <= max_counts) &
    (adata_8um.obs["n_genes_by_counts"] >= min_genes) &
    (adata_8um.obs["pct_counts_mt"] <= max_mt) &
    (~adata_8um.obs["spatial_outlier"].astype(bool))
)
adata_8um = adata_8um[keep].copy()
min_cells = max(int(0.001 * adata_8um.shape[0]), 3)
sc.pp.filter_genes(adata_8um, min_cells=min_cells)

print(f"Filtering: {n_before:,} -> {adata_8um.shape[0]:,} bins "
      f"(removed {n_before - adata_8um.shape[0]:,}, "
      f"{100*(n_before - adata_8um.shape[0])/n_before:.1f}%)")

# Load precomputed normalized + clustered data.
# I ran normalization, PCA, UMAP, and Leiden clustering separately and saved
# the result. The bin count differs slightly from the filtering above because
# the precomputed file was generated from a separate run. Here is what those
# steps would look like:
#
# sc.pp.normalize_total(adata_8um, target_sum=1e4)
# sc.pp.log1p(adata_8um)
# sc.pp.highly_variable_genes(adata_8um, n_top_genes=2000, flavor="seurat_v3")
# adata_8um = adata_8um[:, adata_8um.var["highly_variable"]].copy()
# sc.pp.scale(adata_8um, max_value=10)
# sc.tl.pca(adata_8um, n_comps=50)
# sc.pp.neighbors(adata_8um, n_neighbors=15, n_pcs=30)
# sc.tl.umap(adata_8um)
# sc.tl.leiden(adata_8um, resolution=0.5)

adata_8um = sc.read_h5ad(PRECOMPUTED_DIR / "crc_p1_annotated_50k.h5ad")
print(f"Loaded precomputed annotated data: {adata_8um.shape}")
print(f"Leiden clusters: {adata_8um.obs['leiden'].nunique()}")

# UMAP colored by QC metrics + Leiden
fig, axes = plt.subplots(1, 4, figsize=(22, 5))
for ax, metric, cmap in zip(axes[:3],
    ["total_counts", "n_genes_by_counts", "pct_counts_mt"],
    ["viridis", "viridis", "RdYlBu_r"]):
    sc_p = ax.scatter(adata_8um.obsm["X_umap"][:, 0], adata_8um.obsm["X_umap"][:, 1],
                      c=adata_8um.obs[metric], s=0.3, cmap=cmap,
                      edgecolors="none", rasterized=True)
    ax.set_title(metric); plt.colorbar(sc_p, ax=ax, shrink=0.7)

for cl in sorted(adata_8um.obs["leiden"].unique(), key=int):
    mask = adata_8um.obs["leiden"] == cl
    axes[3].scatter(adata_8um.obsm["X_umap"][mask, 0], adata_8um.obsm["X_umap"][mask, 1],
                    s=0.3, label=cl, alpha=0.5, rasterized=True)
axes[3].set_title("Leiden clusters")
axes[3].legend(markerscale=5, fontsize=7, ncol=2, loc="best")
plt.tight_layout()
plt.savefig(FIG_DIR / "crc_umap_qc_leiden.png", dpi=150, bbox_inches="tight")
plt.show()

### What this means

* Combined spatial + global filtering removed ~22% of bins. The UMAP shows clear cluster structure after filtering.

* High-count/high-mt bins cluster together in the bottom-right. This is likely biological (metabolically active tumor epithelium) rather than pure artifact, since the same region corresponds to tumor cell types when we check annotations next.

## Ground-Truth Cell Type Annotations

These annotations come from the **original study group** ([de Oliveira et al., Nature Genetics 2025](https://www.nature.com/articles/s41588-025-02193-3)), not from us. They used unsupervised clustering and deconvolution on the full dataset. We use their labels as our reference to evaluate our own annotation approach.

**UnsupervisedL1** gives 10 broad categories: Tumor, Fibroblast, T cells, B cells, Myeloid, Endothelial, Smooth Muscle, Intestinal Epithelial, Neuronal, Unknown.

In [ ]:
# Visualize ground-truth UnsupervisedL1
ct_col = "UnsupervisedL1"
has_label = adata_8um.obs[ct_col].notna()
adata_labeled = adata_8um[has_label].copy()
print(f"Bins with {ct_col}: {has_label.sum():,} / {adata_8um.shape[0]:,}")

cell_types = sorted(adata_labeled.obs[ct_col].dropna().unique())
palette = dict(zip(cell_types, plt.cm.tab20(np.linspace(0, 1, len(cell_types)))))

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ct in cell_types:
    mask = (adata_labeled.obs[ct_col] == ct).values
    axes[0].scatter(adata_labeled.obsm["X_umap"][mask, 0], adata_labeled.obsm["X_umap"][mask, 1],
                    s=0.5, label=ct, alpha=0.5, rasterized=True, color=palette[ct])
axes[0].legend(markerscale=8, fontsize=8, loc="best", framealpha=0.8)
axes[0].set_title(f"UMAP: {ct_col} (ground truth)"); axes[0].set_xlabel("UMAP 1"); axes[0].set_ylabel("UMAP 2")

coords = adata_labeled.obsm["spatial"]
for ct in cell_types:
    mask = (adata_labeled.obs[ct_col] == ct).values
    axes[1].scatter(coords[mask, 0], coords[mask, 1],
                    s=0.3, label=ct, alpha=0.4, rasterized=True, color=palette[ct])
axes[1].legend(markerscale=10, fontsize=8, loc="best", framealpha=0.8)
axes[1].set_title(f"Spatial: {ct_col}"); axes[1].set_aspect("equal"); axes[1].invert_yaxis(); axes[1].axis("off")

plt.tight_layout()
plt.savefig(FIG_DIR / "crc_ground_truth_unsupervised.png", dpi=150, bbox_inches="tight")
plt.show()

# Proportions
fig, ax = plt.subplots(figsize=(10, 5))
props = adata_labeled.obs[ct_col].value_counts(normalize=True).sort_values(ascending=True)
colors = [palette.get(ct, "gray") for ct in props.index]
ax.barh(range(len(props)), props.values, color=colors)
ax.set_yticks(range(len(props))); ax.set_yticklabels(props.index)
ax.set_xlabel("Proportion"); ax.set_title(f"{ct_col} proportions")
for i, v in enumerate(props.values):
    ax.text(v + 0.005, i, f"{v:.1%}", va="center", fontsize=9)
plt.tight_layout()
plt.savefig(FIG_DIR / "crc_ground_truth_proportions.png", dpi=150, bbox_inches="tight")
plt.show()

### What this means

* **UMAP**: Cell types separate well. Tumor occupies a large cluster in the bottom-center/right, fibroblast sits adjacent, intestinal epithelial on the left. The high-count/high-mt UMAP clusters correspond to tumor and epithelial compartments, confirming the elevated mt% there is biological.

* **Spatial**: Cell types map to anatomically sensible regions. Tumor forms a contiguous mass, fibroblasts surround it (stroma), intestinal epithelium lines the mucosal surface, immune cells are interspersed.

* **Proportions**: Tumor and Unknown are both large fractions. The exact proportions depend on this particular subsample of the full dataset, so they may shift slightly between runs. The high Unknown fraction (~20%) reflects bins where multiple cells overlap at 8 um resolution and the original clustering could not assign a confident label. Not all bins from the full dataset matched our subsample's barcodes, so annotation coverage is less than 100%.

## Our Marker-Based Annotations

Now we annotate bins ourselves using canonical CRC marker gene sets. This is a common approach when you do not have ground-truth labels.

**How `sc.tl.score_genes` works:** For each marker panel, scanpy computes a per-bin enrichment score. It takes the mean expression of the marker genes, subtracts the mean expression of a reference set of randomly selected genes with similar expression levels, and returns the difference. This controls for overall expression level so that a bin is not called "Tumor" just because it has high total counts. Each bin then gets assigned to whichever panel scored highest. Bins where the top score is too low (<0.05) or too close to the second-best (margin <0.02) are labeled "Low_confidence."

We use the same broad categories as the ground truth (UnsupervisedL1) so the comparison is direct.

In [ ]:
%%time
# Marker-based annotation using broad cell types matching ground truth
print("Scoring marker panels...")

marker_sets = {
    "Tumor":                  ["EPCAM", "KRT8", "KRT18", "KRT19", "KRT20", "CEACAM5", "MUC1"],
    "Fibroblast":             ["COL1A1", "COL1A2", "COL3A1", "DCN", "LUM", "COL6A1"],
    "Endothelial":            ["PECAM1", "VWF", "EMCN", "KDR", "RAMP2", "PLVAP"],
    "Myeloid":                ["LST1", "TYROBP", "FCER1G", "CTSS", "AIF1", "C1QC", "C1QB"],
    "T cells":                ["CD3D", "CD3E", "TRBC1", "TRBC2", "NKG7", "IL7R", "LTB"],
    "B cells":                ["MS4A1", "CD79A", "CD79B", "MZB1", "JCHAIN", "IGKC", "CD74"],
    "Smooth Muscle":          ["ACTA2", "MYH11", "DES", "TAGLN", "CNN1", "ACTG2"],
    "Intestinal Epithelial":  ["MUC2", "TFF3", "FCGBP", "CLCA1", "ZG16", "AGR2",
                               "FABP1", "FABP2", "ALPI", "VIL1", "SIS"],
}

present_markers = {}
for ct, genes in marker_sets.items():
    present = [g for g in genes if g in adata_8um.var_names]
    present_markers[ct] = present
    print(f"  {ct:25s}  {len(present):>2}/{len(genes)} genes  {present}")

score_cols = []
for ct, genes in present_markers.items():
    if len(genes) >= 2:
        score_col = f"{ct}_score"
        sc.tl.score_genes(adata_8um, gene_list=genes, score_name=score_col, use_raw=False)
        score_cols.append(score_col)

score_df = adata_8um.obs[score_cols].copy()
adata_8um.obs["cell_type_marker"] = score_df.idxmax(axis=1).str.replace("_score", "", regex=False)

max_score = score_df.max(axis=1)
sorted_scores = np.sort(score_df.values, axis=1)
second_score = sorted_scores[:, -2] if sorted_scores.shape[1] > 1 else np.zeros(len(score_df))
margin = max_score.values - second_score
adata_8um.obs.loc[(max_score < 0.05) | (margin < 0.02), "cell_type_marker"] = "Low_confidence"

print("\nMarker-based cell type distribution:")
print(adata_8um.obs["cell_type_marker"].value_counts().to_string())

### What this means

* Each bin gets scored against every marker panel and assigned to the highest scorer. The distribution shows our annotation results.

* **Why so many fibroblasts?** Fibroblast markers (COL1A1, DCN, LUM) are among the most broadly expressed genes in solid tissue. They are not specific to fibroblasts: cancer-associated fibroblasts, myofibroblasts, some smooth muscle cells, and even some tumor cells express these collagens. At 8 um resolution where bins can overlap multiple cells, a bin at the tumor-stroma interface may capture both tumor and stromal transcripts, and whichever panel wins by a slim margin gets the label. This is a known limitation of marker-based scoring on spatial data.

* The comparison against ground truth below will show exactly where these disagreements happen.

In [ ]:
# Marker score heatmap
score_summary = (
    adata_8um.obs.groupby("cell_type_marker")[score_cols]
    .median()
    .rename(columns=lambda x: x.replace("_score", ""))
)
fig, ax = plt.subplots(figsize=(10, 7))
im = ax.imshow(score_summary.values, aspect="auto", cmap="viridis")
ax.set_xticks(np.arange(score_summary.shape[1]))
ax.set_xticklabels(score_summary.columns, rotation=45, ha="right")
ax.set_yticks(np.arange(score_summary.shape[0]))
ax.set_yticklabels(score_summary.index)
ax.set_title("Median marker scores by assigned cell type")
plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.savefig(FIG_DIR / "crc_marker_score_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

### What this means

* **Diagonal**: Most cell types show their highest score on the matching panel (bright diagonal). The scoring is working as intended.

* **Cross-reactivity**: Fibroblast and Tumor bins bleed into each other's columns. At the tumor-stroma interface, bins express both collagen and keratin markers.

* **Low_confidence**: Flat scores across all panels. These bins genuinely lack clear marker expression.

In [ ]:
# Compare our marker-based annotations against ground truth
has_both = (adata_8um.obs["UnsupervisedL1"].notna()) & (adata_8um.obs["cell_type_marker"] != "Low_confidence")
if has_both.sum() > 0:
    agree = (adata_8um.obs.loc[has_both, "cell_type_marker"] == adata_8um.obs.loc[has_both, "UnsupervisedL1"])
    print(f"Agreement (marker vs ground-truth): {agree.mean():.1%}  ({has_both.sum():,} bins)")

    cm = pd.crosstab(
        adata_8um.obs.loc[has_both, "cell_type_marker"],
        adata_8um.obs.loc[has_both, "UnsupervisedL1"],
        normalize="index",
    )
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", ax=ax)
    ax.set_xlabel("Ground Truth (UnsupervisedL1)")
    ax.set_ylabel("Our Marker-Based Annotation")
    ax.set_title(f"Annotation Agreement: {agree.mean():.1%}")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "crc_annotation_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()

### What this means

* **Overall agreement**: Shows how well our simple marker scoring matches the original study's unsupervised clustering + deconvolution approach. Cell types with highly specific markers (Tumor keratins, Smooth Muscle actins, T cell receptors) match well. Types with broadly expressed markers (Fibroblast collagens, B cell CD74) match poorly.

* **Fibroblast row**: A chunk of our fibroblast-called bins are ground-truth Unknown or Tumor, confirming the fibroblast inflation from non-specific markers.

* This is a realistic result for marker-based scoring on 8 um bins with shallow depth. More sophisticated methods (deconvolution, label transfer from scRNA-seq reference) would improve agreement.

In [ ]:
# Spatial maps of our marker-based annotations
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
ct_col = "cell_type_marker"
cell_types = sorted(adata_8um.obs[ct_col].unique())
palette_m = dict(zip(cell_types, plt.cm.tab20(np.linspace(0, 1, len(cell_types)))))

for ct in cell_types:
    mask = (adata_8um.obs[ct_col] == ct).values
    axes[0].scatter(adata_8um.obsm["X_umap"][mask, 0], adata_8um.obsm["X_umap"][mask, 1],
                    s=0.5, label=ct, alpha=0.5, rasterized=True, color=palette_m[ct])
axes[0].legend(markerscale=8, fontsize=7, loc="best"); axes[0].set_title("UMAP: Our annotations")

coords = adata_8um.obsm["spatial"]
for ct in cell_types:
    mask = (adata_8um.obs[ct_col] == ct).values
    axes[1].scatter(coords[mask, 0], coords[mask, 1],
                    s=0.3, label=ct, alpha=0.4, rasterized=True, color=palette_m[ct])
axes[1].legend(markerscale=10, fontsize=7, loc="best")
axes[1].set_title("Spatial: Our annotations"); axes[1].set_aspect("equal"); axes[1].invert_yaxis(); axes[1].axis("off")
plt.tight_layout()
plt.savefig(FIG_DIR / "crc_marker_cell_type_maps.png", dpi=150, bbox_inches="tight")
plt.show()

### What this means

* Despite moderate quantitative agreement with ground truth, the spatial patterns are reasonable. Tumor fills the lower-right, fibroblast fills the stroma, intestinal epithelium lines the mucosal surface. The spatial architecture is consistent between our annotations and the ground truth, even where the specific labels disagree.

## Neighborhood Analysis

Which cell types sit next to each other in tissue? **Neighborhood enrichment** tests this by comparing the actual frequency of cell type pairs among spatial neighbors to what you would expect by random permutation. If Tumor bins are surrounded by other Tumor bins more often than chance, that is self-enrichment. If T cells avoid the tumor region, that shows up as negative enrichment.

The algorithm:
1. Build a spatial neighbor graph (k=20 nearest bins)
2. Count how often each pair of cell types appears as neighbors
3. Randomly shuffle the cell type labels many times and recount
4. Compare the real counts to the permuted distribution (z-score)

In [ ]:
%%time
# Neighborhood enrichment
ct_key = "UnsupervisedL1"
adata_nhood = adata_8um[adata_8um.obs[ct_key].notna()].copy()
adata_nhood.obs[ct_key] = adata_nhood.obs[ct_key].astype("category")

sq.gr.spatial_neighbors(adata_nhood, n_neighs=20, coord_type="generic")
sq.gr.nhood_enrichment(adata_nhood, cluster_key=ct_key)

fig, ax = plt.subplots(figsize=(8, 7))
sq.pl.nhood_enrichment(adata_nhood, cluster_key=ct_key, ax=ax)
ax.set_title("Neighborhood Enrichment (ground truth labels)")
plt.tight_layout()
plt.savefig(FIG_DIR / "crc_nhood_enrichment.png", dpi=150, bbox_inches="tight")
plt.show()

### What this means

* **Strong self-enrichment** (bright diagonal) for Tumor, Smooth Muscle, and Intestinal Epithelial. These types form contiguous spatial domains.

* **Tumor avoids most other types** (dark off-diagonal entries in the Tumor row/column). Consistent with a dense tumor mass with limited immune infiltration. The Tumor-T cell avoidance suggests immune exclusion, which is relevant for immunotherapy context.

* **Fibroblast-Intestinal Epithelial**: Negative enrichment despite both being abundant. They occupy adjacent but distinct spatial compartments.

In [ ]:
%%time
# Co-occurrence: how does co-localization change with distance?
try:
    sq.gr.co_occurrence(adata_nhood, cluster_key=ct_key)

    co = adata_nhood.uns[f"{ct_key}_co_occurrence"]
    occ = co["occ"]
    interval = co["interval"]

    if len(interval) == occ.shape[-1] + 1:
        x = (interval[:-1] + interval[1:]) / 2
    else:
        x = interval

    cats = list(adata_nhood.obs[ct_key].cat.categories)
    n = len(cats)
    ncols = 5
    nrows = int(np.ceil(n / ncols))
    palette_co = dict(zip(cats, plt.cm.tab10(np.linspace(0, 1, n))))

    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3 * nrows),
                             sharex=True, sharey=True)
    axes_flat = axes.flat if hasattr(axes, "flat") else [axes]

    for i, anchor in enumerate(cats):
        ax = axes_flat[i]
        for j, other in enumerate(cats):
            ax.plot(x, occ[i, j, :], label=other, color=palette_co[other], lw=1.2)
        ax.axhline(1.0, color="gray", ls="--", lw=0.7)
        ax.set_title(f"Anchor: {anchor}", fontsize=9)
        ax.set_xlabel("Distance"); ax.set_ylabel("P(other | anchor)")

    for k in range(n, len(list(axes_flat))):
        axes_flat[k].axis("off")

    handles, labels = axes_flat[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="center right", fontsize=8,
               bbox_to_anchor=(1.02, 0.5), title=ct_key)
    plt.suptitle("Co-occurrence by cell type", fontsize=13, y=1.02)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "crc_co_occurrence.png", dpi=150, bbox_inches="tight")
    plt.show()
except Exception as e:
    print(f"Co-occurrence analysis: {e}")

del adata_nhood; gc.collect()

### What this means

* **Co-occurrence** shows how the probability of finding each cell type changes with distance from an anchor type. Values >1 mean co-localization, <1 means avoidance, decaying toward 1 at large distances.

* **Tumor**: Self-enriched at short range, all other types below 1. Reinforces the immune exclusion pattern.

* **Neuronal**: Massively self-enriched at short range (enteric neurons cluster in the myenteric plexus) with Smooth Muscle as a secondary neighbor.

* **T cells**: Moderate self-enrichment with slight B cell and Myeloid co-occurrence at short range, consistent with immune niches in the stroma.

* The difference from neighborhood enrichment: co-occurrence tells you how the relationship changes with distance. Two types might co-localize as immediate neighbors but avoid each other at longer distances.

## Spatially Variable Genes (SVGs)

Standard differential expression (DE) analysis asks: which genes differ between groups (e.g., Tumor vs. Fibroblast)? SVG detection asks a different question: which genes have **spatially structured expression patterns**, regardless of any grouping?

**Moran's I** measures spatial autocorrelation. For each gene, it compares expression at each bin to expression at neighboring bins. If nearby bins tend to have similar expression (spatial clustering), the gene gets a high I value. If expression is random with respect to location, I is near zero.

This is useful because:
- It finds spatially patterned genes without requiring prior cell type labels
- It can detect genes with gradients, boundary effects, or regional enrichment that DE would miss
- It validates whether known markers are actually spatially structured in this tissue

In [ ]:
%%time
# SVG detection (subsampled for speed)
n_svg = min(10_000, adata_8um.shape[0])
print(f"Subsampling to {n_svg:,} bins for SVG analysis...")
adata_svg = sc.pp.subsample(adata_8um, n_obs=n_svg, copy=True)
sq.gr.spatial_neighbors(adata_svg, n_neighs=20, coord_type="generic")

if "highly_variable" in adata_svg.var.columns:
    adata_svg_hvg = adata_svg[:, adata_svg.var["highly_variable"]].copy()
else:
    adata_svg_hvg = adata_svg.copy()

print(f"Running Moran's I on {adata_svg_hvg.shape[1]} genes...")
sq.gr.spatial_autocorr(adata_svg_hvg, mode="moran", n_jobs=1)

morans = adata_svg_hvg.uns["moranI"].sort_values("I", ascending=False)
print(f"\nTop 20 SVGs:")
print(morans.head(20)[["I", "pval_norm"]].to_string())

# Plot a curated set of top SVGs representing different tissue compartments
# Pick from top-ranked genes that represent mucosal, muscle, stromal, and tumor compartments
plot_genes = []
compartment_genes = {
    "mucosal": ["PIGR", "MUC2", "MUC12", "FCGBP", "ZG16"],
    "muscle": ["DES", "MYH11"],
    "stromal": ["COL1A1", "COL3A1"],
    "tumor": ["CEACAM5"],
}
for compartment, candidates in compartment_genes.items():
    for g in candidates:
        if g in morans.index and len(plot_genes) < 8:
            plot_genes.append(g)
# Fill remaining slots from top Moran's I genes not already included
for g in morans.index:
    if g not in plot_genes and len(plot_genes) < 8:
        plot_genes.append(g)

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
coords = adata_svg.obsm["spatial"]

for ax, gene in zip(axes.flat, plot_genes):
    if gene in adata_svg.var_names:
        vals = adata_svg[:, gene].X
        expr = vals.toarray().flatten() if sparse.issparse(vals) else np.asarray(vals).flatten()
    else:
        expr = np.zeros(adata_svg.shape[0])
    i_val = morans.loc[gene, "I"] if gene in morans.index else 0
    sc_p = ax.scatter(coords[:, 0], coords[:, 1], c=expr, s=1, cmap="magma",
                      edgecolors="none", rasterized=True)
    ax.set_title(f"{gene}\n(I={i_val:.3f})", fontsize=10)
    ax.set_aspect("equal"); ax.invert_yaxis(); ax.axis("off")
    plt.colorbar(sc_p, ax=ax, shrink=0.6)

plt.suptitle("Spatially Variable Genes", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(FIG_DIR / "crc_svgs.png", dpi=150, bbox_inches="tight")
plt.show()

del adata_svg, adata_svg_hvg; gc.collect()

### What this means

* The plotted genes represent different tissue compartments: **PIGR/MUC2/MUC12/FCGBP** are mucosal epithelium markers concentrated in the upper-left glandular region. **DES/MYH11** are smooth muscle, localized to the muscularis layer. **COL1A1/COL3A1** are fibroblast/stromal genes. **CEACAM5** is a tumor epithelial marker concentrated in the tumor mass.

* Signal is sparse (most bins are dark) because dropout is high at 8 um depth. Moran's I captures the spatial coherence of the non-zero bins even with heavy dropout.

* All p-values are 0 with 10K bins, so the I value itself is what matters for ranking, not statistical significance.

## Cell-Cell Communication

Ligand-receptor (L-R) analysis tests whether specific signaling molecule pairs are expressed at higher levels between spatially adjacent cell types than expected by chance. We use squidpy's permutation-based approach, restricted to the 6 most abundant cell types and subsampled to 5K bins for memory.

In [ ]:
%%time
# L-R analysis
ct_key = "UnsupervisedL1"
adata_lr_src = adata_8um[adata_8um.obs[ct_key].notna()].copy()

top_cts = adata_lr_src.obs[ct_key].value_counts().head(6).index.tolist()
print(f"Restricting L-R analysis to top 6 cell types: {top_cts}")
adata_lr_src = adata_lr_src[adata_lr_src.obs[ct_key].isin(top_cts)].copy()

n_lr = min(5_000, adata_lr_src.shape[0])
adata_lr = sc.pp.subsample(adata_lr_src, n_obs=n_lr, copy=True)
del adata_lr_src
adata_lr.obs[ct_key] = adata_lr.obs[ct_key].astype("category")

sq.gr.spatial_neighbors(adata_lr, n_neighs=20, coord_type="generic")

print(f"Running L-R analysis on {adata_lr.shape[0]:,} bins, 50 permutations...")
sq.gr.ligrec(adata_lr, cluster_key=ct_key, n_perms=50, use_raw=False)
print("Done.")

# Plot significant L-R pairs (compact)
try:
    fig, ax = plt.subplots(figsize=(8, 6))
    sq.pl.ligrec(adata_lr, cluster_key=ct_key,
                 pvalue_threshold=0.01, means_range=(0.5, np.inf),
                 remove_empty_interactions=True,
                 remove_nonsig_interactions=True,
                 swap_axes=True, ax=ax)
    plt.savefig(FIG_DIR / "crc_ligrec.png", dpi=150, bbox_inches="tight")
    plt.show()
except Exception as e:
    print(f"L-R plot: {e}")
    # Try with more permissive filters
    try:
        sq.pl.ligrec(adata_lr, cluster_key=ct_key,
                     pvalue_threshold=0.05, means_range=(0.3, np.inf),
                     remove_empty_interactions=True)
        plt.savefig(FIG_DIR / "crc_ligrec.png", dpi=150, bbox_inches="tight")
        plt.show()
    except Exception as e2:
        print(f"L-R plot with relaxed filters: {e2}")

del adata_lr; gc.collect()

### What this means

* Each dot in the plot represents a ligand-receptor pair between two cell type groups. Dot size reflects the mean expression level; color reflects the p-value (darker = more significant).

* **What to look for**: Collagen-integrin pairs between fibroblasts and other types indicate ECM remodeling. Chemokine-receptor pairs between immune cells and tumor suggest active immune recruitment or suppression. Growth factor signaling at the tumor-stroma interface (e.g., FGF, HGF) suggests paracrine signaling supporting tumor growth.

* This is a quick screen (5K bins, 50 permutations). The small sample and few permutations mean some real interactions may be missed, and some hits may not replicate with more permutations. For publication-quality results, increase both. The key value here is hypothesis generation: which cell type pairs are likely communicating, and through which pathways?

## Marker Validation

Validate the ground-truth annotations by checking whether canonical markers are expressed in the expected cell types.

In [ ]:
# Dotplot of canonical markers by ground-truth cell type
validation = {
    "Tumor":     ["EPCAM", "KRT20", "KRT8"],
    "Fibroblast": ["COL1A1", "DCN", "LUM"],
    "T cells":   ["CD3D", "CD3E"],
    "B cells":   ["MS4A1", "CD79A"],
    "Myeloid":   ["TYROBP", "C1QC", "AIF1"],
    "Endothelial": ["PECAM1", "VWF"],
    "Smooth Muscle": ["ACTA2", "MYH11"],
    "Intestinal Epithelial": ["MUC2", "TFF3", "FABP1"],
}

valid = {}
for ct, genes in validation.items():
    present = [g for g in genes if g in adata_8um.var_names]
    if present:
        valid[ct] = present

adata_gt = adata_8um[adata_8um.obs["UnsupervisedL1"].notna()].copy()
if valid:
    try:
        sc.pl.dotplot(adata_gt, var_names=valid, groupby="UnsupervisedL1",
                      use_raw=False, standard_scale="var",
                      title="Canonical markers by ground-truth cell type")
        plt.savefig(FIG_DIR / "crc_marker_dotplot.png", dpi=150, bbox_inches="tight")
        plt.show()
    except Exception as e:
        print(f"Dotplot: {e}")
        flat = [g for gs in valid.values() for g in gs]
        sc.pl.dotplot(adata_gt, var_names=flat, groupby="UnsupervisedL1", use_raw=False)
        plt.show()

### What this means

* Each marker group shows strongest expression (largest, darkest dots) in the expected cell type row. This confirms the ground-truth labels are biologically sensible.

* **Fibroblast markers** (COL1A1/DCN/LUM) are broadly expressed across many cell types, which is why our marker-based annotation over-called fibroblasts.

* **Immune markers** (CD3D, MS4A1, TYROBP) show small dots even in their correct rows, confirming gene dropout at 8 um depth is the reason marker-based annotation underdetected immune cells.

In [ ]:
# Periphery labels: Tumor vs Tissue vs 50 um border
if "Periphery" in adata_8um.obs.columns and adata_8um.obs["Periphery"].notna().any():
    fig, ax = plt.subplots(figsize=(9, 8))
    coords = adata_8um.obsm["spatial"]
    periph_colors = {"Tumor": "red", "Tissue": "steelblue", "50 micron": "gold"}

    for label, color in periph_colors.items():
        mask = (adata_8um.obs["Periphery"] == label).values
        if mask.any():
            ax.scatter(coords[mask, 0], coords[mask, 1], s=0.3, c=color,
                       label=f"{label} ({mask.sum():,})", alpha=0.4, rasterized=True)

    ax.legend(markerscale=10, fontsize=10)
    ax.set_title("Tumor Periphery Classification")
    ax.set_aspect("equal"); ax.invert_yaxis(); ax.axis("off")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "crc_periphery.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Periphery labels not available.")

### What this means

* Three zones: **Tumor core** (red), **normal Tissue** (blue), and a **50 um border strip** (yellow) at the tumor-stroma interface.

* The yellow strip traces the invasive front. This is the most biologically interesting zone for studying immune infiltration, epithelial-to-mesenchymal transition, and tumor-stroma crosstalk. The border bins are a natural set for differential expression between tumor core vs. invasive margin.

## Summary

1. **QC**: Removed ~22% of bins using per-bin thresholds (UMIs, genes, mt%) plus spatial outlier detection. Both approaches catch different problems.
2. **Ground truth**: The original study's annotations show well-separated cell types with anatomically sensible spatial patterns.
3. **Our annotations**: Marker-based scoring using broad cell types that match ground truth categories. Agreement is moderate, limited mainly by non-specific fibroblast markers and immune gene dropout at shallow depth.
4. **Spatial analysis**: Neighborhood enrichment and co-occurrence confirm tumor self-clustering with immune exclusion. SVGs match expected tissue compartment markers.
5. **Cell-cell communication**: L-R analysis identifies candidate signaling interactions between spatially proximal cell types.

For detailed explanations of each step, see the [comprehensive tutorial](https://colab.research.google.com/github/stevenpastor/spatial_transcriptomics_resources/blob/main/notebook/visium_hd_crc_p1_tutorial.ipynb).